# H2S-NN Graph Classification Notebook

This notebook prepares the workspace and launches the H2S-NN graph-classification training entrypoint:

- `scripts/run_graph_classification_folds.py`
- `configs/experiments/mutag_h2snn.yaml`
- `configs/experiments/proteins_h2snn.yaml`
- `configs/experiments/enzymes_h2snn.yaml`
- `configs/experiments/nci1_h2snn.yaml`

Metrics reported: Macro-F1 and Micro-F1, per fold (matching HetSheaf's node/graph
classification protocol). PROTEINS/ENZYMES/NCI1 reuse the same precomputed 10-fold
splits (`folds/*_splits.json`) as the legacy CSNN baseline, for direct comparability;
MUTAG has no precomputed split and gets a fresh stratified 10-fold split generated
and cached on first run.

Pooling is a swappable ablation axis (mirrors HetSheaf's own pooling ablation,
Table 1): `meanpool` (baseline floor), `grampool` (O(d)-invariant, exploits that
H2SNN's restriction maps are always orthogonal), `s2pool` (GL(d)-invariant, per
`S2Pool_proposal.md`).

Recommended runtime: GPU.

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/H2S-NN_test")
else:
    PROJECT_DIR = Path.cwd()

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_DIR}. Update PROJECT_DIR in this cell if needed."
    )

os.chdir(PROJECT_DIR)
print(f"Working directory: {PROJECT_DIR}")
print(f"Python executable: {sys.executable}")

In [ ]:
%pip install -r requirements.txt

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch version: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE != "cuda":
    print("Warning: training will be much slower on CPU.")

In [ ]:
DATASET = "mutag"  # one of: "mutag", "proteins", "enzymes", "nci1"
POOLING_TYPE = "meanpool"  # one of: "meanpool", "grampool", "s2pool"
NUM_FOLDS = 10

CONFIG_BY_DATASET = {
    "mutag": "configs/experiments/mutag_h2snn.yaml",
    "proteins": "configs/experiments/proteins_h2snn.yaml",
    "enzymes": "configs/experiments/enzymes_h2snn.yaml",
    "nci1": "configs/experiments/nci1_h2snn.yaml",
}

CONFIG_PATH = CONFIG_BY_DATASET[DATASET]
EXP_NAME = f"{DATASET}_h2snn_notebook_{POOLING_TYPE}"
print(f"Config: {CONFIG_PATH}")
print(f"Pooling: {POOLING_TYPE}")
print(f"Experiment name: {EXP_NAME}")

In [ ]:
import os
import subprocess
import sys

command = [
    sys.executable,
    "-u",
    "scripts/run_graph_classification_folds.py",
    "--config",
    CONFIG_PATH,
    "--exp_name",
    EXP_NAME,
    "--device",
    DEVICE,
    "--pooling_type",
    POOLING_TYPE,
    "--num_folds",
    str(NUM_FOLDS),
]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

print("Running:", " ".join(command))
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")

In [ ]:
from pathlib import Path
import json

print(f"Aggregating best test metrics across {NUM_FOLDS} folds for {EXP_NAME}\n")

test_accs, test_macro_f1s, test_micro_f1s = [], [], []
for fold_index in range(NUM_FOLDS):
    exp_dir = Path("experiments") / f"{EXP_NAME}_fold{fold_index}"
    best_path = exp_dir / f"best_test_metrics_fold{fold_index}.json"
    if not best_path.exists():
        print(f"fold {fold_index}: no best_test_metrics file found at {best_path}")
        continue
    record = json.loads(best_path.read_text(encoding="utf-8"))
    test_accs.append(record["test_acc"])
    test_macro_f1s.append(record["test_macro_f1"])
    test_micro_f1s.append(record["test_micro_f1"])
    print(f"fold {fold_index}: test_acc={record['test_acc']:.4f} "
          f"test_macro_f1={record['test_macro_f1']:.4f} test_micro_f1={record['test_micro_f1']:.4f}")

if test_accs:
    n = len(test_accs)
    print(f"\nMean over {n} folds: "
          f"test_acc={sum(test_accs)/n:.4f} "
          f"test_macro_f1={sum(test_macro_f1s)/n:.4f} "
          f"test_micro_f1={sum(test_micro_f1s)/n:.4f}")